# LAB·X4 · Two loose ends

**Hardware:** Colab TPU runtime (Runtime -> Change runtime type -> TPU).

LAB·X2 published a timing that turned out to be nothing: `--xla_disable_hlo_passes=fusion` named a pass the TPU pipeline does not register, so XLA ignored the flag and both runs compiled the identical module. Retracting that left two smaller questions, and this lab answers both.

The first is about measurement. Two identical compilations differed by 26 percent, which is a fact about the harness. If the gap follows the position in the sequence rather than anything about the program, then a first process on a fresh chip is simply slower than the ones after it, and every timing this course publishes needs to account for that.

The second is about vocabulary. If `fusion` is the wrong name on this backend, what is the right one? The pass list this same driver dumps has the answer in it, and this lab greps it rather than guessing again.

## Question one: does the gap follow the order?

Six identical processes, no flags anywhere, each compiling and timing the same attention. Nothing differs between them except when they ran. If the first one is slower and the rest agree with each other, the answer is position, and LAB·X2's 26 percent was the cost of being first.

**your prediction:** will the first process be slower, faster, or indistinguishable?

In [ ]:
# Colab TPU runtime only
import subprocess, sys, textwrap

TIMING_SCRIPT = textwrap.dedent("""
import os
os.environ["XLA_FLAGS"] = "{flags}"
import time
import jax
import jax.numpy as jnp

def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v

S, D = 2048, 128
q = jax.random.normal(jax.random.key(0), (S, D), jnp.bfloat16)
k = jax.random.normal(jax.random.key(1), (S, D), jnp.bfloat16)
v = jax.random.normal(jax.random.key(2), (S, D), jnp.bfloat16)

fn = jax.jit(attend)
fn(q, k, v).block_until_ready()
times = []
for _ in range(20):
    t0 = time.perf_counter()
    fn(q, k, v).block_until_ready()
    times.append(time.perf_counter() - t0)
print(sorted(times)[len(times) // 2] * 1e6)
""")

def timed_subprocess(flags=""):
    out = subprocess.run([sys.executable, "-c", TIMING_SCRIPT.format(flags=flags)],
                         capture_output=True, text=True, timeout=600)
    if out.returncode != 0:
        print(out.stderr[-2000:])
        raise RuntimeError("subprocess failed; see stderr above")
    return float(out.stdout.strip().splitlines()[-1])

runs = [timed_subprocess() for _ in range(6)]
for i, us in enumerate(runs, 1):
    print(f"process {i}: {us:7.1f} us")

rest = runs[1:]
print()
print(f"first:            {runs[0]:.1f} us")
print(f"median of others: {sorted(rest)[len(rest) // 2]:.1f} us")
print(f"spread of others: {min(rest):.1f} to {max(rest):.1f} us")
print(f"first vs others:  {runs[0] / (sorted(rest)[len(rest) // 2]):.2f}x")

A ratio near `1.0` with the others tightly grouped says position does not matter and LAB·X2's gap was ordinary variance. A first process meaningfully slower than a tight group after it says the opposite, and every timing in this course that compares two fresh processes has to discard the first or interleave them.

**write it down:** the chip, the six numbers, and which reading you got.

## Question two: what is the fusion stage actually called?

The dump names every pass it runs. If a stage on this backend does fusion, its name is in that list, and the reason the flag did nothing is that the name is not the word `fusion`. This grep is the whole investigation.

In [ ]:
# Colab TPU runtime only
import os
os.environ["XLA_FLAGS"] = "--xla_dump_to=/tmp/xla-x4 --xla_dump_hlo_pass_re=.*"

import re
import jax
import jax.numpy as jnp

assert jax.devices()[0].platform == "tpu", "Runtime -> Change runtime type -> TPU"

def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v

x = jnp.ones((64, 64))
jax.jit(attend).lower(x, x, x).compile()

files = sorted(f for f in os.listdir("/tmp/xla-x4") if "jit_attend" in f)
pattern = re.compile(r"jit_attend\.(?:cl_\d+\.)?(\d{4}\..+)\.txt$")
steps = sorted({m.group(1) for f in files if (m := pattern.search(f))})
print(f"{len(steps)} pass steps dumped")

hits = [s for s in steps if "fus" in s.lower()]
print(f"{len(hits)} mention fusion:")
for s in hits:
    print("  ", s)

Each dumped step name reads `NNNN.<pipeline>.after_<pass>.before_<pass>`, so the names to try are the ones sitting in the `after_` and `before_` positions, not the pipeline label in front of them.

**your prediction:** pick the one name you would pass to `--xla_disable_hlo_passes` and write it down before running the next cell.

In [ ]:
# Colab TPU runtime only
import subprocess, sys, textwrap

COMPILE_SCRIPT = textwrap.dedent("""
import os
os.environ["XLA_FLAGS"] = "{flags}"
import jax
import jax.numpy as jnp
def attend(q, k, v):
    s = q @ k.T / jnp.sqrt(jnp.float32(q.shape[-1]))
    return jax.nn.softmax(s) @ v
x = jnp.ones((2048, 128), jnp.bfloat16)
print(jax.jit(attend).lower(x, x, x).compile().as_text())
""")

def compiled_text(flags=""):
    out = subprocess.run([sys.executable, "-c", COMPILE_SCRIPT.format(flags=flags)],
                         capture_output=True, text=True, timeout=600)
    if out.returncode != 0:
        return None
    return out.stdout

# Harvest every distinct pass name the dump mentioned around a fusion stage,
# then find which ones XLA will actually accept and act on.
candidates = []
for s in hits:
    for part in re.findall(r"(?:after|before)_([\w-]+)", s):
        if "fus" in part.lower() and part not in candidates:
            candidates.append(part)
print("candidate pass names:", candidates or "none found in the dump")

baseline = compiled_text("")
for name in candidates:
    text = compiled_text(f"--xla_disable_hlo_passes={name}")
    if text is None:
        print(f"  {name}: compile failed with this flag")
    elif text != baseline:
        print(f"  {name}: CHANGED the module  <- this is the real name")
    else:
        print(f"  {name}: ignored, module identical")

A name that changes the module is the one LAB·X2 should have used, and the fusion comparison that chapter 5 wanted becomes possible for the first time. Time both versions with the same harness above and the answer is finally about fusion rather than about a flag.

A run where nothing changes the module is also an answer: on this backend fusion may not be a separately disableable stage at all, in which case the honest way to measure its value is a different experiment, not a better flag.

## paste-back

```
chip:
six process timings (in order):
first vs others ratio:
pass steps dumped:
names mentioning fusion:
name that changed the module:
```

Both answers land on the site: the ordering result decides whether this course's timing harness needs fixing, and the pass name reopens chapter 5's fusion question with the right tool.

## mark it run

Chapter 05 (kernels.rudrite.com/xla/fusion) holds the retraction this lab is chasing, and chapter 04 (kernels.rudrite.com/xla/pipeline) is where the dump and its naming are taught.